In [1]:
import torch
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import json
import re
import os


# --- 1. 모델 로드 ---
# Hugging Face 토큰이 필요한 경우 미리 로그인합니다.
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")

print("LLM (Llama 3.1)을 로딩합니다...")
try:
    # Llama 3.1 모델 로드
    # 이 모델은 캡션 생성과 질문 의도 분석 두 가지 작업에 모두 사용됩니다.

    device_num = 0
    model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    llm_pipe = pipeline(
        "text-generation",
        model=model_id,
        model_kwargs={"torch_dtype": torch.bfloat16}, # torch_dtype은 model_kwargs 안에 넣는 것이 권장됩니다.
        device_map=f"cuda:{device_num}",
    )
    # llm_pipe = pipeline(
    #     "text-generation",
    #     model="meta-llama/Meta-Llama-3.1-8B-Instruct",
    #     model_kwargs={"torch_dtype": torch.bfloat16},
    #     device_map="0",
    # )
    print("LLM 로딩이 완료되었습니다.")
except Exception as e:
    print(f"LLM 모델 로딩 중 오류가 발생했습니다: {e}")
    print("GPU 메모리가 부족하거나, 인터넷 연결, Hugging Face 토큰을 확인해주세요.")
    exit()

# --- 2. 경로 설정 ---
# 입력 CSV 파일 경로
input_csv_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
# 전처리된 결과를 저장할 CSV 파일 경로
output_csv_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_by_llm.csv"

# --- 3. 프롬프트 템플릿 정의 ---

# 3-1. 캡션 생성을 위한 프롬프트
caption_prompt_template = """
You are an AI assistant that creates concise, affirmative, and descriptive captions from multiple-choice options.
Your task is to look at each option and turn it into a short, positive caption, as if it were describing a photo.
Completely ignore the structure of the question (e.g., "Which is not...", "What is the purpose...") and only use the content of the options (A, B, C, D) to generate the captions.

### Example 1
Question: Which of the following foods is not present in the image?
A: Pizza
B: Blueberries
C: Salmon
D: Avocado

### Output 1
A: A photo of a pizza.
B: A photo of blueberries.
C: A photo of salmon.
D: A photo of an avocado.

### Example 2
Question: What might be the purpose of the person's workout in the image?
A: Building muscle and strength
B: Practicing for a marathon
C: Training for a cycling race
D: Preparing for a swimming competition

### Output 2
A: A person building muscle and strength.
B: A person practicing for a marathon.
C: A person training for a cycling race.
D: A person preparing for a swimming competition.

### Your Task
Question: {question}
A: {option_A}
B: {option_B}
C: {option_C}
D: {option_D}

### Output
"""

# 3-2. 질문 의도 분석을 위한 프롬프트
logic_prompt_template = """
You are a logical analysis expert. Your task is to analyze a visual question and determine the correct evaluation metric.
When a Visual Question Answering model compares an image to four text captions (A, B, C, D), should it look for the caption with the highest "yes" score or the highest "no" score to find the correct answer?

- For questions asking what IS in the image, the correct answer will have the highest "yes" score.
- For questions asking what IS NOT in the image, or which statement is FALSE, the correct answer will have the highest "no" score.

Analyze the following question and respond ONLY with a JSON object containing one key, "target_answer", with a value of either "yes" or "no".

### Example 1
Question: What is the woman in the picture doing?
JSON:
{{"target_answer": "yes"}}

### Example 2
Question: Which of the following foods is not present in the image?
JSON:
{{"target_answer": "no"}}

### Your Task
Question: {question}
JSON:
"""


# --- 4. LLM 호출 함수 정의 ---

def generate_captions(row, llm_pipeline):
    """LLM을 사용하여 한 행에 대한 4개의 긍정형 캡션을 생성합니다."""
    prompt = caption_prompt_template.format(
        question=row['Question'],
        option_A=row['A'], option_B=row['B'], option_C=row['C'], option_D=row['D']
    )
    messages = [{"role": "user", "content": prompt}]
    try:
        outputs = llm_pipeline(messages, max_new_tokens=256, temperature=0.1)
        response_text = outputs[0]["generated_text"][-1]['content'].strip()
        
        # 'A: ...', 'B: ...' 형식의 출력을 파싱하여 텍스트만 추출
        captions = [line.split(':', 1)[1].strip() if ':' in line else line for line in response_text.split('\n')]
        
        if len(captions) == 4:
            return captions
        else: # 예외 처리
            print(f"\nWarning: 행 ID {row.get('ID', 'N/A')}의 캡션 파싱 실패. Raw: '{response_text}'")
            return [row['A'], row['B'], row['C'], row['D']] # 파싱 실패 시 원본 보기 사용
            
    except Exception as e:
        print(f"\nError in generate_captions for ID {row.get('ID', 'N/A')}: {e}")
        return [None, None, None, None]

def determine_evaluation_logic(question, llm_pipeline):
    """LLM을 사용하여 질문의 평가 기준('yes' 또는 'no')을 결정합니다."""
    prompt = logic_prompt_template.format(question=question)
    messages = [{"role": "user", "content": prompt}]
    try:
        outputs = llm_pipeline(messages, max_new_tokens=20, temperature=0.1)
        response_text = outputs[0]["generated_text"][-1]['content']
        
        # LLM 응답에서 JSON 부분만 정확히 추출
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            logic = json.loads(json_match.group(0))
            return logic.get("target_answer", "yes") # JSON 파싱 성공 시 값 반환
        
        print(f"\nWarning: 질문 '{question[:30]}...'의 로직 JSON 파싱 실패. Raw: '{response_text}'")
        return "yes" # 파싱 실패 시 기본값 'yes' 반환
        
    except Exception as e:
        print(f"\nError in determine_evaluation_logic for question '{question[:30]}...': {e}")
        return "yes"

# --- 5. 메인 실행 루프 ---
try:
    df = pd.read_csv(input_csv_path)
    print(f"'{input_csv_path}'에서 {len(df)}개의 행을 읽었습니다. 전처리를 시작합니다.")
except FileNotFoundError:
    print(f"입력 파일 '{input_csv_path}'을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 결과를 저장할 리스트 초기화
all_captions_A, all_captions_B, all_captions_C, all_captions_D = [], [], [], []
all_target_answers = []

# tqdm을 사용하여 전체 진행 상황 표시
for index, row in tqdm(df.iterrows(), total=len(df), desc="LLM Preprocessing"):
    # 1. 캡션 생성
    captions = generate_captions(row, llm_pipe)
    all_captions_A.append(captions[0])
    all_captions_B.append(captions[1])
    all_captions_C.append(captions[2])
    all_captions_D.append(captions[3])
    
    # 2. 평가 로직 결정
    target_answer = determine_evaluation_logic(row['Question'], llm_pipe)
    all_target_answers.append(target_answer)

# --- 6. 결과 종합 및 저장 ---
print("\n전처리가 완료되었습니다. 결과를 CSV 파일로 저장합니다.")

# 기존 DataFrame에 새로운 열 추가
df['caption_A'] = all_captions_A
df['caption_B'] = all_captions_B
df['caption_C'] = all_captions_C
df['caption_D'] = all_captions_D
df['target_answer'] = all_target_answers

# 결과를 새로운 CSV 파일로 저장
df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업 완료! 최종 결과가 '{output_csv_path}' 파일에 저장되었습니다.")
print("\n--- 최종 전처리 결과 미리보기 (상위 5개) ---")
# 보기 좋게 일부 열만 선택하여 출력
preview_columns = ['ID', 'Question', 'caption_A', 'caption_B', 'target_answer']
print(df[preview_columns].head())

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM (Llama 3.1)을 로딩합니다...


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]
Device set to use cuda:0


LLM 로딩이 완료되었습니다.
'/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'에서 60개의 행을 읽었습니다. 전처리를 시작합니다.


LLM Preprocessing:   0%|          | 0/60 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
LLM Preprocessing:   2%|▏         | 1/60 [00:01<01:24,  1.44s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
LLM Preprocessing:   3%|▎         | 2/60 [00:02<01:10,  1.22s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
LLM Preprocessing:   5%|▌         | 3/60 [00:03<00:59,  1.04s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
LLM Preprocessing:   7%|▋         | 4/60 [00:04<01:00,  1.08s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to 


전처리가 완료되었습니다. 결과를 CSV 파일로 저장합니다.

✅ 모든 작업 완료! 최종 결과가 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_by_llm.csv' 파일에 저장되었습니다.

--- 최종 전처리 결과 미리보기 (상위 5개) ---
         ID                                           Question  \
0  TEST_000  What might be the purpose of the person's work...   
1  TEST_001          What color is the shoe rack in the image?   
2  TEST_002    What might be the deer's behavior in the image?   
3  TEST_003  Considering the equipment used in the image, w...   
4  TEST_004  What is a common ingredient in Gimbap (Korean ...   

                                caption_A  \
0  A person building muscle and strength.   
1                A sleek black shoe rack.   
2                          A deer eating.   
3            A photo of a Pilates studio.   
4                      A photo of cheese.   

                             caption_B target_answer  
0  A person practicing for a marathon.           yes  
1            A